In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from opacus import PrivacyEngine
import gc
import numpy as np
import torch.nn.functional as F
from DP_SGD_Materials.compute_privacy_sgm import compute_dp_sgd_privacy



transform_fashionmnist = transforms.Compose([
    transforms.Grayscale(num_output_channels=1), 
    transforms.Resize((32, 32)),               
    transforms.ToTensor(),                       
])

fm_dataset  = datasets.FashionMNIST(root='./data', train=True, transform=transform_fashionmnist, download=True)
fm_loader  = DataLoader(dataset=fm_dataset, batch_size=64, shuffle=False)

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1), 
    transforms.ToTensor(),                       
])

cifar_dataset = datasets.CIFAR10(root='./data', train=True, transform=transform, download=True)

private_subset = cifar_dataset

np.random.seed(42)
fm_indices = np.random.choice(len(fm_dataset), size=2000, replace=False)
public_subset = Subset(fm_dataset, fm_indices)

cifar_test_data = datasets.CIFAR10(root='./data', train=False, transform=transform, download=True)
test_loader = DataLoader(dataset=cifar_test_data, batch_size=64, shuffle=False)

train_loader_private = DataLoader(dataset=private_subset, batch_size=64, shuffle=True)
train_loader_public = DataLoader(dataset=public_subset, batch_size=64, shuffle=True)

In [2]:
class Model(nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        self.conv1 = nn.Conv2d(1,6, kernel_size=5, stride=1, padding=2)
        self.tanh1 = nn.Tanh()
        self.pool1 = nn.AvgPool2d(kernel_size=2,  stride=2)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5, stride=1)
        self.tanh2 = nn.Tanh()
        self.pool2 = nn.AvgPool2d(kernel_size=2, stride=2)
        self.conv3 = nn.Conv2d(16, 120, kernel_size=5, stride=1)
        self.tanh3 = nn.Tanh()
        self.fc1 = nn.Linear(480, 84)
        self.tanh4 = nn.Tanh()
        self.fc2 = nn.Linear(84, 10)
    def forward(self, x):
        x = self.pool1(self.tanh1(self.conv1(x)))
        x = self.pool2(self.tanh2(self.conv2(x)))
        x = self.tanh3(self.conv3(x))
        x = x.view(-1, 480)
        x = self.tanh4(self.fc1(x))
        x = self.fc2(x)
        return x
    
model = Model()

In [3]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

privacy_engine = PrivacyEngine()

gc.collect()

/Users/rubenhayrapetyan/Library/Python/3.9/lib/python/site-packages/opacus/privacy_engine.py:96: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(


0

In [4]:
epochs_1 = 5
train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []
for epoch in range(epochs_1):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader_public:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    print(f'Epoch [{epoch + 1}/{epochs_1}]')
    print("Training loss", (running_loss / total))  
    train_losses.append(running_loss / total)
    print("Training accuracy", (correct *100/ total),'%')
    train_accuracies.append(correct * 100 / total)
    gc.collect()

    model.eval()
    running_loss_eval = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss_eval += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print("Testing loss", (running_loss / total))
    test_losses.append(running_loss_eval / total)
    print("Testing accuracy", (correct*100 / total),'%')
    test_accuracies.append(correct * 100 / total)
    gc.collect()

Epoch [1/5]
Training loss 2.2305134201049803
Training accuracy 20.75 %
Testing loss 0.4461026840209961
Testing accuracy 10.19 %
Epoch [2/5]
Training loss 1.5226883096694945
Training accuracy 50.7 %
Testing loss 0.3045376619338989
Testing accuracy 9.79 %
Epoch [3/5]
Training loss 1.0512433795928955
Training accuracy 60.65 %
Testing loss 0.2102486759185791
Testing accuracy 9.92 %
Epoch [4/5]
Training loss 0.8886966972351075
Training accuracy 66.7 %
Testing loss 0.17773933944702147
Testing accuracy 10.6 %
Epoch [5/5]
Training loss 0.7676615934371949
Training accuracy 71.35 %
Testing loss 0.15353231868743897
Testing accuracy 10.05 %


In [5]:
# from opacus.noise_generator import GaussianNoiseGenerator
from opacus.validators import ModuleValidator

epochs_2 = 10
train_losses_2 = []
train_accuracies_2 = []
test_losses_2 = []
test_accuracies_2 = []
model.train()
target_delta = 1e-5
# noise_generator = GaussianNoiseGenerator(
#     noise_multiplier=sigma,
#     sample_rate=sample_rate,  # Required
#     device=device,            # "cuda" or "cpu"
# )
model, optimizer, train_loader_private = privacy_engine.make_private(
    module=model,
    optimizer=optimizer,
    data_loader=train_loader_private,
    epochs=10,
    target_delta=target_delta,
    max_grad_norm=1,
    noise_multiplier=0.0
)
for epoch in range(epochs_2):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader_private:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    print(f'Epoch [{epoch + 1}/{epochs_2}]')
    print("Training loss", (running_loss / total))
    train_losses_2.append(running_loss / total)
    print("Training accuracy", (correct *100/ total),'%')
    train_accuracies_2.append(correct * 100 / total)
    gc.collect()

    # epsilon = privacy_engine.get_epsilon(delta=target_delta)
    # print(f"Privacy Budget (epsilon, delta): ({epsilon:.2f}, {target_delta})")

    model.eval()
    running_loss_eval = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss_eval += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print("Testing loss", (running_loss / total))
    test_losses_2.append(running_loss_eval / total)
    print("Testing accuracy", (correct*100 / total),'%')
    test_accuracies_2.append(correct * 100 / total)

    gc.collect()
    model.eval()
    correct = 0
    total = 0
    eval_loss = 0.0

    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            eval_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

print(f"Test Loss: {eval_loss / total:.4f}, Accuracy: {100 * correct / total:.2f}%")

Epoch [1/10]
Training loss 2.3916091676742908
Training accuracy 15.64914915904105 %
Testing loss 12.10082490568161
Testing accuracy 19.02 %
Epoch [2/10]
Training loss 2.149569120528731
Training accuracy 23.433258821650966 %
Testing loss 10.770201121497154
Testing accuracy 26.21 %
Epoch [3/10]
Training loss 2.0807187342099285
Training accuracy 27.013227194146264 %
Testing loss 10.35074341520071
Testing accuracy 27.7 %
Epoch [4/10]
Training loss 2.043493545341918
Training accuracy 28.70118981074822 %
Testing loss 10.236267867326736
Testing accuracy 28.85 %
Epoch [5/10]
Training loss 2.0077514900990017
Training accuracy 29.86600775427389 %
Testing loss 9.99438614256382
Testing accuracy 29.78 %
Epoch [6/10]
Training loss 1.9793150601442913
Training accuracy 30.815624937174565 %
Testing loss 9.845311040663718
Testing accuracy 31.5 %
Epoch [7/10]
Training loss 1.9469286736144893
Training accuracy 31.844831740422457 %
Testing loss 9.696483566069602
Testing accuracy 32.15 %
Epoch [8/10]
Traini